In [0]:
# Databricks Notebook — OSL Process Matching
# Tablo: hcdap_prod.bronze_hc_tri_rlc.osl_export_290526
# Bu notebook transcript dosyaların hazır olmasa da çalışır.
# Transcript yoksa ADIM 3'ü atla, ADIM 4'ten devam et.

# ─── ADIM 1: Template süreçleri yükle ────────────────────────────────────────
template_df = spark.sql("""
    SELECT
        `Process ID`          AS process_id,
        `Process Name`        AS process_name,
        `Process Description` AS process_desc,
        `Lead Table`          AS lead_table,
        `Process Type`        AS process_type,
        `Data Status`         AS data_status
    FROM hcdap_prod.bronze_hc_tri_rlc.osl_export_290526
    WHERE `Data Status` = 'Productive'
""").toPandas()

# Matching için arama metni: isim + açıklama birleştir
import pandas as pd

def build_search_text(row):
    parts = []
    if row["process_name"] and str(row["process_name"]) != "nan":
        parts.append(str(row["process_name"]))
    if row["process_desc"] and str(row["process_desc"]) not in ["nan", "tbd", "TBD"]:
        parts.append(str(row["process_desc"]))
    return " ".join(parts)

template_df["search_text"] = template_df.apply(build_search_text, axis=1)

print(f"Template süreç sayısı: {len(template_df)}")
print(f"\nÖrnek search_text:")
for _, row in template_df.head(3).iterrows():
    print(f"  [{row['process_id']}] {row['search_text'][:100]}")


# ─── ADIM 2: Lead Table kategorilerini gör ────────────────────────────────────
print("\nLead Table dağılımı:")
print(template_df["lead_table"].value_counts().to_string())


# ─── ADIM 3: Transcript dosyalarını yükle ─────────────────────────────────────
# Transcript dosyalarını DBFS'e yüklediysen bu adımı çalıştır.
# Henüz yüklemediysen bu hücreyi atla — ADIM 4'e git.
#
# Yükleme yolu: Databricks > Catalog > (sol üst) > "+" > Add data > Upload files
# Veya: Data > DBFS > FileStore klasörüne yükle
# Desteklenen format: .txt (Türkçe veya İngilizce fark etmez)

import os

TRANSCRIPT_DIR = "/dbfs/FileStore/transcripts/"
doc_records = []

if os.path.exists(TRANSCRIPT_DIR):
    for filename in os.listdir(TRANSCRIPT_DIR):
        if filename.endswith(".txt"):
            with open(os.path.join(TRANSCRIPT_DIR, filename), "r", encoding="utf-8") as f:
                text = f.read()
            # 300 kelimelik chunk'lara böl
            words = text.split()
            for i in range(0, len(words), 300):
                chunk = " ".join(words[i:i+300])
                doc_records.append({
                    "source_file": filename,
                    "chunk_id": i // 300,
                    "chunk_text": chunk
                })
    print(f"Yüklenen chunk sayısı: {len(doc_records)}")
else:
    print("Transcript klasörü bulunamadı — ADIM 4'e geç.")


# ─── ADIM 4: TF-IDF Matching ──────────────────────────────────────────────────
# Transcript varsa: her chunk'ı template süreçlerle karşılaştır
# Transcript yoksa: template süreçleri kendi aralarında gruplayıp benzer olanları bul

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

results = []

if doc_records:
    # ── Senaryo A: Transcript VAR ─────────────────────────────────────────────
    docs_df = pd.DataFrame(doc_records)
    doc_texts = docs_df["chunk_text"].tolist()
    template_texts = template_df["search_text"].tolist()

    vectorizer = TfidfVectorizer(min_df=1, max_df=0.95, ngram_range=(1,2), sublinear_tf=True)
    all_texts = template_texts + doc_texts
    tfidf_matrix = vectorizer.fit_transform(all_texts)

    template_matrix = tfidf_matrix[:len(template_texts)]
    doc_matrix = tfidf_matrix[len(template_texts):]

    THRESHOLD = 0.08
    TOP_K = 3

    for i, t_row in template_df.iterrows():
        scores = cosine_similarity(template_matrix[i], doc_matrix)[0]
        top_idx = np.argsort(scores)[::-1][:TOP_K]
        for rank, idx in enumerate(top_idx):
            score = scores[idx]
            if score >= THRESHOLD:
                d_row = docs_df.iloc[idx]
                results.append({
                    "process_id":      t_row["process_id"],
                    "process_name":    t_row["process_name"],
                    "lead_table":      t_row["lead_table"],
                    "match_rank":      rank + 1,
                    "similarity_score": round(float(score), 4),
                    "source_file":     d_row["source_file"],
                    "matched_text":    d_row["chunk_text"][:300],
                })

    results_df = pd.DataFrame(results)
    print(f"\n✓ Matching tamamlandı")
    print(f"  Eşleşen süreç: {results_df['process_id'].nunique()} / {len(template_df)}")

else:
    # ── Senaryo B: Transcript YOK — süreçleri kendi aralarında karşılaştır ───
    # Bu, lokasyonun kendi süreçlerini (soru-cevaplardan çıkan) manuel girmeden önce
    # template'in iç yapısını anlamak için kullanışlı.
    print("\nTranscript bulunamadı.")
    print("Lütfen DBFS'e transcript yükle ve ADIM 3'ü tekrar çalıştır.")
    print("\nŞimdilik template içi benzerlik analizi yapılıyor...")

    template_texts = template_df["search_text"].tolist()
    vectorizer = TfidfVectorizer(min_df=1, ngram_range=(1,2), sublinear_tf=True)
    tfidf_matrix = vectorizer.fit_transform(template_texts)
    sim_matrix = cosine_similarity(tfidf_matrix)

    for i, t_row in template_df.iterrows():
        scores = sim_matrix[i]
        scores[i] = 0  # kendisiyle eşleşmeyi dışla
        top_idx = np.argsort(scores)[::-1][:3]
        for rank, idx in enumerate(top_idx):
            score = scores[idx]
            if score >= 0.15:
                similar_row = template_df.iloc[idx]
                results.append({
                    "process_id":       t_row["process_id"],
                    "process_name":     t_row["process_name"],
                    "lead_table":       t_row["lead_table"],
                    "similar_to_id":    similar_row["process_id"],
                    "similar_to_name":  similar_row["process_name"],
                    "similarity_score": round(float(score), 4),
                })

    results_df = pd.DataFrame(results)
    print(f"\nBenzer süreç çifti sayısı: {len(results_df)}")


# ─── ADIM 5: Sonuçları göster ─────────────────────────────────────────────────
display(results_df.sort_values("similarity_score", ascending=False).head(30))


# ─── ADIM 6: Lead Table bazında özet ──────────────────────────────────────────
if "source_file" in results_df.columns:
    print("\n=== Lead Table Bazında Eşleşme ===")
    summary = results_df[results_df["match_rank"]==1].groupby("lead_table").agg(
        eslesme=("process_id","count"),
        ort_skor=("similarity_score","mean")
    ).reset_index().sort_values("ort_skor", ascending=False)
    display(summary)


# ADIM 7 - eski kod silindi, kayıt yeni hücrede yapılıyor
print("Kayıt işlemi yeni hücrede Delta Table olarak yapılıyor.")
# ─── Dosyaları oku ve chunk'lara böl ────────────────────────────────────────
import pandas as pd
import os

TRANSCRIPT_PATH = "/Volumes/hcdap_prod/bronze_hc_tri_rlc/alevsplace/q2e_meeting_transcript.txt"
QA_PATH         = "/Volumes/hcdap_prod/bronze_hc_tri_rlc/alevsplace/Q2E_Questions_filled_V02.csv"

doc_records = []

# Transcript
with open(TRANSCRIPT_PATH, "r", encoding="utf-8") as f:
    text = f.read()

words = text.split()
for i in range(0, len(words), 300):
    chunk = " ".join(words[i:i+300])
    doc_records.append({
        "source_file": "q2e_meeting_transcript.txt",
        "chunk_id":    i // 300,
        "chunk_text":  chunk
    })
print(f"Transcript: {len(text.split())} kelime → {len(doc_records)} chunk")

# Soru-cevap CSV
qa_df = pd.read_csv(QA_PATH, encoding="utf-8-sig", on_bad_lines='skip')
print(f"\nQA kolonları: {qa_df.columns.tolist()}")
print(f"QA satır sayısı: {len(qa_df)}")
display(qa_df.head(3))


# ─── QA satırlarını chunk'a çevir ────────────────────────────────────────────
# Tüm kolonları birleştirerek bir metin oluştur
# (Kolon adlarını gördükten sonra gerekirse aşağıyı güncelle)
def qa_row_to_text(row):
    parts = []
    for val in row:
        v = str(val).strip()
        if v and v.lower() not in ["nan", "none", "tbd", "-", ""]:
            parts.append(v)
    return " ".join(parts)

qa_texts = qa_df.apply(qa_row_to_text, axis=1).tolist()

qa_start = len(doc_records)
for idx, chunk_text in enumerate(qa_texts):
    if len(chunk_text.split()) > 5:  # Çok kısa satırları atla
        doc_records.append({
            "source_file": "Q2E_Questions_filled_V02.csv",
            "chunk_id":    idx,
            "chunk_text":  chunk_text
        })

print(f"\nQA: {len(qa_texts)} satır → {len(doc_records) - qa_start} chunk eklendi")
print(f"Toplam chunk: {len(doc_records)}")


# ─── TF-IDF Matching ─────────────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

docs_df = pd.DataFrame(doc_records)
template_texts = template_df["search_text"].tolist()
doc_texts      = docs_df["chunk_text"].tolist()

vectorizer = TfidfVectorizer(
    min_df=1,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

all_texts    = template_texts + doc_texts
tfidf_matrix = vectorizer.fit_transform(all_texts)

template_matrix = tfidf_matrix[:len(template_texts)]
doc_matrix      = tfidf_matrix[len(template_texts):]

print(f"\n✓ Vektörizasyon tamamlandı — vocabulary: {len(vectorizer.vocabulary_)} kelime")

THRESHOLD = 0.08
TOP_K     = 3
results   = []

for i, t_row in template_df.iterrows():
    scores  = cosine_similarity(template_matrix[i], doc_matrix)[0]
    top_idx = np.argsort(scores)[::-1][:TOP_K]
    for rank, idx in enumerate(top_idx):
        score = scores[idx]
        if score >= THRESHOLD:
            d_row = docs_df.iloc[idx]
            results.append({
                "process_id":       t_row["process_id"],
                "process_name":     t_row["process_name"],
                "lead_table":       t_row["lead_table"],
                "match_rank":       rank + 1,
                "similarity_score": round(float(score), 4),
                "güven":            "YÜKSEK" if score >= 0.35 else "ORTA" if score >= 0.20 else "DÜŞÜK",
                "source_file":      d_row["source_file"],
                "matched_text":     d_row["chunk_text"][:300],
            })

results_df = pd.DataFrame(results)
print(f"\n✓ Matching tamamlandı")
print(f"  Eşleşen süreç : {results_df['process_id'].nunique()} / {len(template_df)}")
print(f"  Toplam eşleşme: {len(results_df)}")
print(f"\nGüven dağılımı:")
print(results_df[results_df['match_rank']==1]['güven'].value_counts().to_string())


# ─── Sonuçları göster ────────────────────────────────────────────────────────
display(
    results_df[results_df["match_rank"] == 1]
    .sort_values("similarity_score", ascending=False)
    .head(30)
)


# ─── Lead Table bazında özet ─────────────────────────────────────────────────
summary = (
    results_df[results_df["match_rank"] == 1]
    .groupby("lead_table")
    .agg(eslesme=("process_id", "count"), ort_skor=("similarity_score", "mean"))
    .reset_index()
    .sort_values("ort_skor", ascending=False)
)
print("\nLead Table bazında özet:")
display(summary)


# ─── Delta Table olarak kaydet ───────────────────────────────────────────────
spark.createDataFrame(results_df).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("hcdap_prod.bronze_hc_tri_rlc.osl_match_results")

print("\n✓ Sonuçlar kaydedildi: hcdap_prod.bronze_hc_tri_rlc.osl_match_results")
print("  SQL Editor'dan görüntüle:")
print("  SELECT * FROM hcdap_prod.bronze_hc_tri_rlc.osl_match_results ORDER BY similarity_score DESC")